<a href="https://colab.research.google.com/github/annatagus/python_aulas_exercicios/blob/main/Aula_spark_20261008.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
# !pip -q install pyspark

from pyspark.sql import SparkSession
spark = (SparkSession.builder
         .appName("Spark-Exercises-Colab")
         .getOrCreate())

spark


In [153]:
from pyspark.sql.functions import col
from pyspark.sql.functions import when

from pyspark.sql.functions import (
    concat,
    upper, lower, length,
    to_date, year, month, round
)

from pyspark.sql.functions import avg, max, min, sum, countDistinct

In [67]:
#Ler os Dados

vendas = spark.read.csv("/content/vendas.csv",header=True,inferSchema=True)

clientes = spark.read.csv("/content/clientes.csv",header=True,inferSchema=True)


In [17]:
vendas.show()

+--------+----------+--------+----------+----------+--------------+------+----------+
|id_venda|      data| produto| categoria|quantidade|preco_unitario|regiao|id_cliente|
+--------+----------+--------+----------+----------+--------------+------+----------+
|       1|2026-02-27| Mochila| Papelaria|         5|            65| Norte|        32|
|       2|2026-03-13| Teclado|Tecnologia|         2|            65| Norte|        76|
|       3|2026-01-17| Estante|Mobiliario|         2|           180| Norte|        28|
|       4|2026-09-16| Teclado|Tecnologia|         4|            65| Norte|        92|
|       5|2026-08-03| Mochila| Papelaria|         8|            65|   Sul|        76|
|       6|2026-01-04| Cadeira|Mobiliario|         7|           220|   Sul|        44|
|       7|2026-03-21| Cadeira|Mobiliario|         6|           220|   Sul|        14|
|       8|2026-07-14|   Mouse|Tecnologia|         6|            25| Norte|        45|
|       9|2026-05-16|  Caneta| Papelaria|         8|  

In [13]:
vendas.printSchema()


root
 |-- id_venda: integer (nullable = true)
 |-- data: date (nullable = true)
 |-- produto: string (nullable = true)
 |-- categoria: string (nullable = true)
 |-- quantidade: integer (nullable = true)
 |-- preco_unitario: integer (nullable = true)
 |-- regiao: string (nullable = true)
 |-- id_cliente: integer (nullable = true)



In [14]:
vendas.describe().show()

+-------+-----------------+-------+----------+------------------+-----------------+------+-----------------+
|summary|         id_venda|produto| categoria|        quantidade|   preco_unitario|regiao|       id_cliente|
+-------+-----------------+-------+----------+------------------+-----------------+------+-----------------+
|  count|             2000|   2000|      2000|              2000|             2000|  1960|             2000|
|   mean|           1000.5|   NULL|      NULL|            4.5925|          213.577|  NULL|          52.2285|
| stddev|577.4945887192364|   NULL|      NULL|2.3065350207401907|324.6232172651765|  NULL|28.86863326767255|
|    min|                1| Agenda|Mobiliario|                 1|                3|Centro|                1|
|    max|             2000|Teclado|Tecnologia|                 8|             1200|   Sul|              100|
+-------+-----------------+-------+----------+------------------+-----------------+------+-----------------+



In [18]:
clientes.show()

+----------+----------+----------+
|id_cliente|      nome|  segmento|
+----------+----------+----------+
|         1| Cliente_1|       VIP|
|         2| Cliente_2|Particular|
|         3| Cliente_3|Particular|
|         4| Cliente_4|       VIP|
|         5| Cliente_5|   Empresa|
|         6| Cliente_6|       VIP|
|         7| Cliente_7|   Empresa|
|         8| Cliente_8|Particular|
|         9| Cliente_9|   Empresa|
|        10|Cliente_10|       VIP|
|        11|Cliente_11|       VIP|
|        12|Cliente_12|       VIP|
|        13|Cliente_13|Particular|
|        14|Cliente_14|   Empresa|
|        15|Cliente_15|   Empresa|
|        16|Cliente_16|       VIP|
|        17|Cliente_17|       VIP|
|        18|Cliente_18|       VIP|
|        19|Cliente_19|       VIP|
|        20|Cliente_20|       VIP|
+----------+----------+----------+
only showing top 20 rows


In [15]:
clientes.printSchema()

root
 |-- id_cliente: integer (nullable = true)
 |-- nome: string (nullable = true)
 |-- segmento: string (nullable = true)



In [16]:
clientes.describe().show()

+-------+------------------+----------+--------+
|summary|        id_cliente|      nome|segmento|
+-------+------------------+----------+--------+
|  count|               100|       100|     100|
|   mean|              50.5|      NULL|    NULL|
| stddev|29.011491975882016|      NULL|    NULL|
|    min|                 1| Cliente_1| Empresa|
|    max|               100|Cliente_99|     VIP|
+-------+------------------+----------+--------+



#Manipulando Colunas

In [28]:
# Select
vendas.select("id_venda", "produto", "categoria", "quantidade").show()


+--------+--------+----------+----------+
|id_venda| produto| categoria|quantidade|
+--------+--------+----------+----------+
|       1| Mochila| Papelaria|         5|
|       2| Teclado|Tecnologia|         2|
|       3| Estante|Mobiliario|         2|
|       4| Teclado|Tecnologia|         4|
|       5| Mochila| Papelaria|         8|
|       6| Cadeira|Mobiliario|         7|
|       7| Cadeira|Mobiliario|         6|
|       8|   Mouse|Tecnologia|         6|
|       9|  Caneta| Papelaria|         8|
|      10|   Mouse|Tecnologia|         5|
|      11|  Caneta| Papelaria|         2|
|      12| Mochila| Papelaria|         2|
|      13|   Mouse|Tecnologia|         8|
|      14|    Mesa|Mobiliario|         6|
|      15| Mochila| Papelaria|         3|
|      16|  Agenda| Papelaria|         8|
|      17| Cadeira|Mobiliario|         1|
|      18|Notebook|Tecnologia|         5|
|      19| Teclado|Tecnologia|         8|
|      20| Mochila| Papelaria|         5|
+--------+--------+----------+----

In [68]:
vendas_2 = vendas.select("produto","categoria","quantidade")

In [69]:
vendas.select(col("quantidade").alias("Unidades")).show()

+--------+
|Unidades|
+--------+
|       5|
|       2|
|       2|
|       4|
|       8|
|       7|
|       6|
|       6|
|       8|
|       5|
|       2|
|       2|
|       8|
|       6|
|       3|
|       8|
|       1|
|       5|
|       8|
|       5|
+--------+
only showing top 20 rows


#Filtrando o DF

In [70]:
# filtrando com Filter

# quantas vendas foram realizadas com mais de 5 unidades
# vendas.filter(col("quantidade")>5).show()
vendas.filter(col("quantidade")>5).count()

# vendas categoria = Papelaria
vendas.filter(col("categoria")=="Papelaria").count()

# vendas de papelaria com mais de 5 unidades
vendas.filter((col("categoria") == "Papelaria") & (col("quantidade") > 5)).count()


275

In [71]:
# filtrando com Where

vendas.where("categoria = 'Papelaria'").count()

vendas_Pap = vendas.where("categoria = 'Papelaria'")


In [72]:
vendas.where("categoria = 'Papelaria' and quantidade > 5").show()

+--------+----------+-------+---------+----------+--------------+------+----------+
|id_venda|      data|produto|categoria|quantidade|preco_unitario|regiao|id_cliente|
+--------+----------+-------+---------+----------+--------------+------+----------+
|       5|2026-08-03|Mochila|Papelaria|         8|            65|   Sul|        76|
|       9|2026-05-16| Caneta|Papelaria|         8|             3| Norte|        69|
|      16|2026-05-06| Agenda|Papelaria|         8|            20|   Sul|        49|
|      26|2026-02-28| Agenda|Papelaria|         6|            20|Centro|        15|
|      35|2026-04-13|Caderno|Papelaria|         7|            12|Centro|        86|
|      36|2026-07-11|Mochila|Papelaria|         8|            65|Lisboa|        16|
|      43|2026-08-09|Mochila|Papelaria|         7|            65|Centro|        53|
|      72|2026-03-18| Agenda|Papelaria|         6|            20| Norte|        75|
|      75|2026-05-08|Mochila|Papelaria|         6|            65| Norte|    

In [45]:
vendas_Pap.show()

+--------+----------+-------+---------+----------+--------------+------+----------+
|id_venda|      data|produto|categoria|quantidade|preco_unitario|regiao|id_cliente|
+--------+----------+-------+---------+----------+--------------+------+----------+
|       1|2026-02-27|Mochila|Papelaria|         5|            65| Norte|        32|
|       5|2026-08-03|Mochila|Papelaria|         8|            65|   Sul|        76|
|       9|2026-05-16| Caneta|Papelaria|         8|             3| Norte|        69|
|      11|2026-07-05| Caneta|Papelaria|         2|             3|   Sul|         6|
|      12|2026-04-27|Mochila|Papelaria|         2|            65|Centro|        30|
|      15|2026-05-17|Mochila|Papelaria|         3|            65| Norte|        69|
|      16|2026-05-06| Agenda|Papelaria|         8|            20|   Sul|        49|
|      20|2026-08-23|Mochila|Papelaria|         5|            65|   Sul|        18|
|      26|2026-02-28| Agenda|Papelaria|         6|            20|Centro|    

#Criar Novas Colunas

In [73]:
vendas.withColumn("TotalFaturacao", col("quantidade")*col("preco_unitario")).show()

+--------+----------+--------+----------+----------+--------------+------+----------+--------------+
|id_venda|      data| produto| categoria|quantidade|preco_unitario|regiao|id_cliente|TotalFaturacao|
+--------+----------+--------+----------+----------+--------------+------+----------+--------------+
|       1|2026-02-27| Mochila| Papelaria|         5|            65| Norte|        32|           325|
|       2|2026-03-13| Teclado|Tecnologia|         2|            65| Norte|        76|           130|
|       3|2026-01-17| Estante|Mobiliario|         2|           180| Norte|        28|           360|
|       4|2026-09-16| Teclado|Tecnologia|         4|            65| Norte|        92|           260|
|       5|2026-08-03| Mochila| Papelaria|         8|            65|   Sul|        76|           520|
|       6|2026-01-04| Cadeira|Mobiliario|         7|           220|   Sul|        44|          1540|
|       7|2026-03-21| Cadeira|Mobiliario|         6|           220|   Sul|        14|      

In [74]:
vendas = vendas.withColumn("TotalFaturacao", col("quantidade") * col("preco_unitario")) # para atualizar a nova coluna no spacename


In [75]:
vendas.show()

+--------+----------+--------+----------+----------+--------------+------+----------+--------------+
|id_venda|      data| produto| categoria|quantidade|preco_unitario|regiao|id_cliente|TotalFaturacao|
+--------+----------+--------+----------+----------+--------------+------+----------+--------------+
|       1|2026-02-27| Mochila| Papelaria|         5|            65| Norte|        32|           325|
|       2|2026-03-13| Teclado|Tecnologia|         2|            65| Norte|        76|           130|
|       3|2026-01-17| Estante|Mobiliario|         2|           180| Norte|        28|           360|
|       4|2026-09-16| Teclado|Tecnologia|         4|            65| Norte|        92|           260|
|       5|2026-08-03| Mochila| Papelaria|         8|            65|   Sul|        76|           520|
|       6|2026-01-04| Cadeira|Mobiliario|         7|           220|   Sul|        44|          1540|
|       7|2026-03-21| Cadeira|Mobiliario|         6|           220|   Sul|        14|      

In [87]:
# Case When

vendas = vendas.withColumn("CategoriaVenda",
                  when(vendas["TotalFaturacao"] >= 1000, "Ouro")
                  .when(vendas["TotalFaturacao"] >= 500, "Prata")
                  .when(vendas["TotalFaturacao"] >= 100, "Bronze")
                  .otherwise("PequenoTicket")
                  )

In [88]:
vendas.show()

+--------+----------+--------+----------+----------+--------------+------+----------+--------------+--------------+
|id_venda|      data| produto| categoria|quantidade|preco_unitario|regiao|id_cliente|TotalFaturacao|CategoriaVenda|
+--------+----------+--------+----------+----------+--------------+------+----------+--------------+--------------+
|       1|2026-02-27| Mochila| Papelaria|         5|            65| Norte|        32|           325|        Bronze|
|       2|2026-03-13| Teclado|Tecnologia|         2|            65| Norte|        76|           130|        Bronze|
|       3|2026-01-17| Estante|Mobiliario|         2|           180| Norte|        28|           360|        Bronze|
|       4|2026-09-16| Teclado|Tecnologia|         4|            65| Norte|        92|           260|        Bronze|
|       5|2026-08-03| Mochila| Papelaria|         8|            65|   Sul|        76|           520|         Prata|
|       6|2026-01-04| Cadeira|Mobiliario|         7|           220|   Su

#Funções de Texto

In [92]:
vendas = vendas.withColumn("ProdutoUpper", upper(vendas["produto"]))

In [93]:
vendas.show()

+--------+----------+--------+----------+----------+--------------+------+----------+--------------+--------------+------------+
|id_venda|      data| produto| categoria|quantidade|preco_unitario|regiao|id_cliente|TotalFaturacao|CategoriaVenda|ProdutoUpper|
+--------+----------+--------+----------+----------+--------------+------+----------+--------------+--------------+------------+
|       1|2026-02-27| Mochila| Papelaria|         5|            65| Norte|        32|           325|        Bronze|     MOCHILA|
|       2|2026-03-13| Teclado|Tecnologia|         2|            65| Norte|        76|           130|        Bronze|     TECLADO|
|       3|2026-01-17| Estante|Mobiliario|         2|           180| Norte|        28|           360|        Bronze|     ESTANTE|
|       4|2026-09-16| Teclado|Tecnologia|         4|            65| Norte|        92|           260|        Bronze|     TECLADO|
|       5|2026-08-03| Mochila| Papelaria|         8|            65|   Sul|        76|           5

# Group by

In [102]:
vendas.groupBy("Regiao").count().show()

+------+-----+
|Regiao|count|
+------+-----+
|  NULL|   40|
|   Sul|  507|
|Lisboa|  494|
| Norte|  485|
|Centro|  474|
+------+-----+



In [103]:
vendas.groupBy("Regiao").sum("TotalFaturacao").show()

+------+-------------------+
|Regiao|sum(TotalFaturacao)|
+------+-------------------+
|  NULL|              28881|
|   Sul|             501496|
|Lisboa|             449133|
| Norte|             493737|
|Centro|             472372|
+------+-------------------+



In [119]:
vendas.groupBy("Regiao").agg(
    avg("TotalFaturacao").alias("MediaFaturamento"),
    sum("TotalFaturacao").alias("SomaFaturamento"),
    max("TotalFaturacao").alias("MaxFaturamento"),
    min("TotalFaturacao").alias("MinFaturamento")).show()

+------+------------------+---------------+--------------+--------------+
|Regiao|  MediaFaturamento|SomaFaturamento|MaxFaturamento|MinFaturamento|
+------+------------------+---------------+--------------+--------------+
|  NULL|           722.025|          28881|          3600|             3|
|   Sul| 989.1439842209073|         501496|          9600|             3|
|Lisboa| 909.1761133603239|         449133|          9600|             3|
| Norte|1018.0144329896907|         493737|          9600|             3|
|Centro| 996.5654008438819|         472372|          9600|             3|
+------+------------------+---------------+--------------+--------------+



In [120]:
vendas.groupBy("Regiao").agg(
    round(avg("TotalFaturacao"),2).alias("MediaFaturamento"),
    sum("TotalFaturacao").alias("SomaFaturamento"),
    max("TotalFaturacao").alias("MaxFaturamento"),
    min("TotalFaturacao").alias("MinFaturamento")).orderBy(col("SomaFaturamento").desc()).show()

+------+----------------+---------------+--------------+--------------+
|Regiao|MediaFaturamento|SomaFaturamento|MaxFaturamento|MinFaturamento|
+------+----------------+---------------+--------------+--------------+
|   Sul|          989.14|         501496|          9600|             3|
| Norte|         1018.01|         493737|          9600|             3|
|Centro|          996.57|         472372|          9600|             3|
|Lisboa|          909.18|         449133|          9600|             3|
|  NULL|          722.03|          28881|          3600|             3|
+------+----------------+---------------+--------------+--------------+



# Joins

In [122]:
vendas_clientes = vendas.join(clientes, vendas.id_cliente == clientes.id_cliente, "inner")

In [123]:
vendas_clientes.show()

+--------+----------+--------+----------+----------+--------------+------+----------+--------------+--------------+------------+----------+----------+----------+
|id_venda|      data| produto| categoria|quantidade|preco_unitario|regiao|id_cliente|TotalFaturacao|CategoriaVenda|ProdutoUpper|id_cliente|      nome|  segmento|
+--------+----------+--------+----------+----------+--------------+------+----------+--------------+--------------+------------+----------+----------+----------+
|       1|2026-02-27| Mochila| Papelaria|         5|            65| Norte|        32|           325|        Bronze|     MOCHILA|        32|Cliente_32|       VIP|
|       2|2026-03-13| Teclado|Tecnologia|         2|            65| Norte|        76|           130|        Bronze|     TECLADO|        76|Cliente_76|Particular|
|       3|2026-01-17| Estante|Mobiliario|         2|           180| Norte|        28|           360|        Bronze|     ESTANTE|        28|Cliente_28|       VIP|
|       4|2026-09-16| Teclad

# Guardar dados

In [130]:
vendas_clientes_limpo = vendas_clientes.drop("id_cliente", "id_venda")

vendas_clientes_limpo.write.mode("overwrite").parquet("/content/vendas_clientes.parquet")

In [128]:
vendas_clientes.write.mode("overwrite").csv("/content/vendas_clientes.csv")

# Usando SQL

In [133]:
vendas_clientes_limpo.createOrReplaceTempView("vendas_clientes_limpo")

In [134]:
spark.sql("select * from vendas_clientes_limpo").show()

+----------+--------+----------+----------+--------------+------+--------------+--------------+------------+----------+----------+
|      data| produto| categoria|quantidade|preco_unitario|regiao|TotalFaturacao|CategoriaVenda|ProdutoUpper|      nome|  segmento|
+----------+--------+----------+----------+--------------+------+--------------+--------------+------------+----------+----------+
|2026-02-27| Mochila| Papelaria|         5|            65| Norte|           325|        Bronze|     MOCHILA|Cliente_32|       VIP|
|2026-03-13| Teclado|Tecnologia|         2|            65| Norte|           130|        Bronze|     TECLADO|Cliente_76|Particular|
|2026-01-17| Estante|Mobiliario|         2|           180| Norte|           360|        Bronze|     ESTANTE|Cliente_28|       VIP|
|2026-09-16| Teclado|Tecnologia|         4|            65| Norte|           260|        Bronze|     TECLADO|Cliente_92|Particular|
|2026-08-03| Mochila| Papelaria|         8|            65|   Sul|           520|   

#Perguntas

**Parte 1 — Conhecer os dados**

1 Quantas transações existem no conjunto de dados?

2 Quais são as categorias de produtos comercializadas?

3 Quantas vendas foram realizadas na região Norte?

4 Existem valores nulos ou informações em falta? Em que colunas?

**Parte 2 — Análise comercial**

*Intermédio*

*Transformações, agregações e KPIs.*

5 Qual é a faturação total da empresa?

6 Qual é o ticket médio por transação?

7 Qual é a categoria que mais contribui para a faturação?

8 Quais são os cinco produtos mais vendidos em quantidade e em valor?

9 Qual região apresenta maior faturação e qual tem o maior ticket médio?

**Parte 3 — Comportamento dos clientes**

*Intermédio*

*Joins e análise entre tabelas.*

10 Qual dos segmentos (Particular, Empresa ou VIP) gera maior faturação?

11 Quantos clientes distintos realizaram compras em cada segmento?

12 Quais são os dez clientes com maior valor acumulado em compras?

13 Existem clientes sem compras registadas? Quantos?


In [155]:
# 1.
vendas.select(countDistinct("id_venda")).show()

+------------------------+
|count(DISTINCT id_venda)|
+------------------------+
|                    2000|
+------------------------+



In [144]:
# 2.
vendas.groupBy("categoria").count().show()

+----------+-----+
| categoria|count|
+----------+-----+
| Papelaria|  688|
|Mobiliario|  644|
|Tecnologia|  668|
+----------+-----+



In [157]:
# 3.
# vendas.groupBy("Regiao").count().show() # não usar count e show simultanea/
vendas_n = vendas.where("Regiao = 'Norte'")
vendas_n.count()

485

In [192]:
# 4.

# Calcula a soma de nulos por coluna de forma compacta
vendas_nulos = vendas.select([sum(when(col(c).isNull(), 1).otherwise(0)).alias(c) for c in vendas.columns]).collect()[0]

# Filtra apenas as colunas onde a contagem é maior que 0
colunas_com_nulos = [c for c in vendas.columns if vendas_nulos[c] > 0]

print("Colunas com valores nulos:", colunas_com_nulos)
vendas_nulos

Colunas com valores nulos: ['regiao']


Row(id_venda=0, data=0, produto=0, categoria=0, quantidade=0, preco_unitario=0, regiao=40, id_cliente=0, TotalFaturacao=0, CategoriaVenda=0, ProdutoUpper=0)

In [183]:
vendas.filter(col("regiao").isNull()).show()

+--------+----------+---------+----------+----------+--------------+------+----------+--------------+--------------+------------+
|id_venda|      data|  produto| categoria|quantidade|preco_unitario|regiao|id_cliente|TotalFaturacao|CategoriaVenda|ProdutoUpper|
+--------+----------+---------+----------+----------+--------------+------+----------+--------------+--------------+------------+
|      50|2026-07-25|  Teclado|Tecnologia|         1|            65|  NULL|        22|            65| PequenoTicket|     TECLADO|
|     100|2026-07-16|   Agenda| Papelaria|         8|            20|  NULL|        52|           160|        Bronze|      AGENDA|
|     150|2026-06-01|  Teclado|Tecnologia|         4|            65|  NULL|        47|           260|        Bronze|     TECLADO|
|     200|2026-10-01|  Mochila| Papelaria|         7|            65|  NULL|        41|           455|        Bronze|     MOCHILA|
|     250|2026-06-16|  Cadeira|Mobiliario|         7|           220|  NULL|        77|    

In [193]:
# Calcula a soma de nulos por coluna de forma compacta
clientes_nulos = clientes.select([sum(when(col(c).isNull(), 1).otherwise(0)).alias(c) for c in clientes.columns]).collect()[0]

# Filtra apenas as colunas onde a contagem é maior que 0
colunas_com_nulos = [c for c in clientes.columns if clientes_nulos[c] > 0]

print("Colunas com valores nulos:", colunas_com_nulos)
clientes_nulos

Colunas com valores nulos: []


Row(id_cliente=0, nome=0, segmento=0)